### Connecting FRED Database to Pull CPI, PPI, GDP, etc. 

In [ ]:
import pandas as pd, numpy as np
from pandas_datareader import data as pdr
import requests

# ── FRED release dates for high-impact events ──
# Each FRED "release" has an id; we pull its historical release dates.
# Release times are standardized (ET): 8:30 for CPI/NFP/PPI/PCE/Retail, 10:00 ISM, 14:00 FOMC.
FRED_RELEASES = {
    10:  ("CPI",        "08:30"),   
    50:  ("NFP",        "08:30"),   
    46:  ("PPI",        "08:30"),  
    54:  ("PCE",        "08:30"),   
    53:  ("GDP",        "08:30"),  
    9:   ("RetailSales","08:30"),   
}
FRED_KEY = "95d04b388119b66203089daf9dbe295f"   # free at fred.stlouisfed.org/docs/api/api_key.html

def fred_release_dates(release_id, start="2021-01-01", quarterly=False):
    url = (f"https://api.stlouisfed.org/fred/release/dates"
           f"?release_id={release_id}&api_key={FRED_KEY}&file_type=json"
           f"&realtime_start={start}&include_release_dates_with_no_data=false&limit=1000")
    j = requests.get(url).json()
    dates = pd.to_datetime([d["date"] for d in j.get("release_dates", [])]).sort_values()
    s = dates.to_series()
    if quarterly:
        # GDP: 3 estimates per quarter -> keep only the FIRST (advance) per quarter
        return s.groupby([dates.year, dates.quarter]).min().dt.date
    else:
        return s.groupby([dates.year, dates.month]).min().dt.date

events = []
for rid, (name, t) in FRED_RELEASES.items():
    is_gdp = (name == "GDP")
    for d in fred_release_dates(rid, quarterly=is_gdp):
        ts = pd.Timestamp(f"{d} {t}", tz="America/New_York")
        if ts >= pd.Timestamp("2021-01-01", tz="America/New_York"):
            events.append({"event": name, "time": ts})

# ── FOMC dates (hardcoded — 8/year, decision at 14:00 ET) ──
FOMC = ["2021-01-27","2021-03-17","2021-04-28","2021-06-16","2021-07-28","2021-09-22","2021-11-03","2021-12-15",
        "2022-01-26","2022-03-16","2022-05-04","2022-06-15","2022-07-27","2022-09-21","2022-11-02","2022-12-14",
        "2023-02-01","2023-03-22","2023-05-03","2023-06-14","2023-07-26","2023-09-20","2023-11-01","2023-12-13",
        "2024-01-31","2024-03-20","2024-05-01","2024-06-12","2024-07-31","2024-09-18","2024-11-07","2024-12-18",
        "2025-01-29","2025-03-19","2025-05-07","2025-06-18","2025-07-30","2025-09-17","2025-10-29","2025-12-10",
        "2026-01-28","2026-03-18","2026-04-29","2026-06-17"]
for d in FOMC:
    events.append({"event": "FOMC", "time": pd.Timestamp(f"{d} 14:00", tz="America/New_York")})

ev = pd.DataFrame(events).sort_values("time").reset_index(drop=True)
print(f"{len(ev)} high-impact events 2021-2026")
print(ev.event.value_counts())

398 high-impact events 2021-2026
event
NFP            67
CPI            67
RetailSales    67
PPI            66
PCE            64
FOMC           44
GDP            23
Name: count, dtype: int64


### Sample Check on News Dates

In [3]:
# spot-check: CPI should be ~mid-month, GDP ~late in quarter-end months, FOMC on known dates
print("CPI dates (should be ~10th-15th):")
print(ev[ev.event=="CPI"].time.head(4).dt.strftime("%Y-%m-%d %H:%M").tolist())
print("GDP dates (should be late Jan/Apr/Jul/Oct = advance estimates):")
print(ev[ev.event=="GDP"].time.head(4).dt.strftime("%Y-%m-%d").tolist())

CPI dates (should be ~10th-15th):
['2021-01-13 08:30', '2021-02-08 08:30', '2021-03-10 08:30', '2021-04-13 08:30']
GDP dates (should be late Jan/Apr/Jul/Oct = advance estimates):
['2021-01-28', '2021-04-29', '2021-07-29', '2021-10-28']
